# Thematic coding using extracted interview data

## Setup

In [ ]:
!pip install groq pandas openpyxl python-dotenv -q

In [ ]:
import os
import json
import time
import pandas as pd
import numpy as np
from collections import Counter
from itertools import combinations
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

os.environ["GROQ_API_KEY"] = "hidden for privacy"

client = Groq(api_key=os.environ["GROQ_API_KEY"])

## Load answers

In [ ]:
with open("output/interview_answers.json", "r", encoding="utf-8") as f:
    records = json.load(f)

df = pd.DataFrame(records)
cols = ["_source_file"] + [c for c in df.columns if c != "_source_file"]
df = df[cols]

question_cols = [c for c in df.columns if c != "_source_file"]

## Label themes

In [ ]:
THEME_CATEGORIES = [
    "time_constraints",
    "body_image",
    "social_media_influence",
    "lack_of_knowledge",
    "low_motivation",
    "gym_intimidation",
    "social_support",
    "nutrition_energy",
    "cost_access",
    "mental_health",
    "positive_self_image",
    "intrinsic_motivation",
    "extrinsic_motivation",
    "autonomy_preference",
    "past_athletic_identity",
]

THEME_LABELS = {t: t.replace("_", " ").title() for t in THEME_CATEGORIES}

## Sentiment scoring and thematic coding using Groq API

In [ ]:
CODING_SYSTEM_PROMPT = f"""You are a qualitative research coding assistant. You will receive a set of interview answers from one participant about gym barriers, body image, and exercise habits.

For EACH answer, provide:
1. **themes**: A list of applicable theme codes from ONLY these categories: {json.dumps(THEME_CATEGORIES)}
   - Assign 1-4 themes per answer. Only assign themes that are clearly supported by the answer.
   - If the answer is "NOT ADDRESSED", assign an empty list.

2. **sentiment**: A score from -2 to +2:
   -2 = strongly negative (distress, avoidance, shame)
   -1 = somewhat negative (frustration, mild discomfort)
    0 = neutral or purely factual
   +1 = somewhat positive (mild optimism, some confidence)
   +2 = strongly positive (enthusiasm, empowerment, confidence)
   If "NOT ADDRESSED", use null.

3. **summary_tag**: A 2-5 word label capturing the essence of the answer (e.g., "time-strapped but motivated", "confident gym-goer", "social media aware").
   If "NOT ADDRESSED", use null.

Respond ONLY with valid JSON. The JSON should be an object where each key is the question text, and each value is an object with "themes", "sentiment", and "summary_tag".
"""


def code_participant(participant_answers: dict) -> dict:
    """Send one participant's answers to Groq for thematic coding."""
    answers_text = json.dumps(participant_answers, indent=2)
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": CODING_SYSTEM_PROMPT},
                    {"role": "user", "content": f"Here are the participant's answers:\n\n{answers_text}\n\nCode each answer with themes, sentiment, and a summary tag."},
                ],
                temperature=0.1,
                max_tokens=4096,
                response_format={"type": "json_object"},
            )
            raw = response.choices[0].message.content.strip()
            return json.loads(raw)
        except json.JSONDecodeError:
            if attempt < 2:
                time.sleep(2)
            else:
                raise
        except Exception as e:
            if "429" in str(e):
                wait = 10 * (attempt + 1)
                time.sleep(wait)
            else:
                raise

In [ ]:
all_codings = {}

for i, (_, row) in enumerate(df.iterrows()):
    name = row["_source_file"]
    participant_answers = {q: row[q] for q in question_cols if q in row}
    try:
        coding = code_participant(participant_answers)
        all_codings[name] = coding
        print("SUCCESS")
    except Exception as e:
        print("ERROR")
    if i < len(df) - 1:
        time.sleep(3)

## Flatten to table with one row per participant per question

In [ ]:
coded_rows = []

for participant, codings in all_codings.items():
    orig_row = df[df["_source_file"] == participant].iloc[0]
    for question in question_cols:
        original_answer = orig_row.get(question, "NOT ADDRESSED")
        coding_entry = codings.get(question, {})
        if not coding_entry:
            for k, v in codings.items():
                if k[:40].lower() == question[:40].lower():
                    coding_entry = v
                    break
        if isinstance(coding_entry, dict):
            themes = coding_entry.get("themes", [])
            sentiment = coding_entry.get("sentiment", None)
            summary_tag = coding_entry.get("summary_tag", None)
        else:
            themes, sentiment, summary_tag = [], None, None
        coded_rows.append({
            "participant": participant,
            "question": question,
            "answer": original_answer,
            "themes": ", ".join(themes) if themes else "",
            "theme_count": len(themes),
            "sentiment": sentiment,
            "summary_tag": summary_tag,
            **{f"theme_{t}": (1 if t in themes else 0) for t in THEME_CATEGORIES}
        })

coded_df = pd.DataFrame(coded_rows)

## From here on, mostly Claude Code to help with sorting and overview

## 6. Analysis: Barrier categories

Which themes came up most across all participants?

In [ ]:
# Count how many participants mentioned each theme (across all their answers)
theme_binary_cols = [f"theme_{t}" for t in THEME_CATEGORIES]

# Per-participant: did they mention this theme at least once?
participant_themes = coded_df.groupby("participant")[theme_binary_cols].max()

theme_prevalence = participant_themes.sum().sort_values(ascending=False)
theme_prevalence.index = [THEME_LABELS[t.replace("theme_", "")] for t in theme_prevalence.index]

print("Theme prevalence (# of participants who mentioned each theme):\n")
for theme, count in theme_prevalence.items():
    bar = "█" * int(count)
    print(f"  {theme:<28} {bar} {int(count)}/{len(df)}")

# Total mentions (not just binary)
print("\n" + "="*60)
total_mentions = coded_df[theme_binary_cols].sum().sort_values(ascending=False)
total_mentions.index = [THEME_LABELS[t.replace("theme_", "")] for t in total_mentions.index]

print("\nTotal mentions across all answers:\n")
for theme, count in total_mentions.items():
    if count > 0:
        print(f"  {theme:<28} {int(count)} mentions")

## 7. Analysis: Sentiment overview

Heatmap-ready data: sentiment score per question per participant.

In [ ]:
# Pivot: rows = questions, columns = participants, values = sentiment
sentiment_pivot = coded_df.pivot_table(
    index="question",
    columns="participant",
    values="sentiment",
    aggfunc="mean"  # handles any duplicates by averaging
)

# Reorder questions to match interview order
ordered_qs = [q for q in question_cols if q in sentiment_pivot.index]
sentiment_pivot = sentiment_pivot.reindex(ordered_qs)

# Short labels for display
short_labels = [f"Q{i+1}: {q[:55]}..." if len(q) > 55 else f"Q{i+1}: {q}" for i, q in enumerate(ordered_qs)]
sentiment_display = sentiment_pivot.copy()
sentiment_display.index = short_labels

print("Sentiment scores per question per participant (-2 to +2):\n")

# Use plain formatting instead of Styler (avoids duplicate index issues)
display(sentiment_display.round(1).fillna("—"))

# Average sentiment per question
print("\nAverage sentiment per question:\n")
avg_sentiment = sentiment_pivot.mean(axis=1)
for i, (q, s) in enumerate(zip(ordered_qs, avg_sentiment)):
    label = f"Q{i+1}: {q[:60]}..." if len(q) > 60 else f"Q{i+1}: {q}"
    if pd.isna(s):
        print(f"  —     {label}")
    else:
        emoji = "🟢" if s > 0.5 else "🔴" if s < -0.5 else "🟡"
        print(f"  {emoji} {s:+.1f}  {label}")

## 8. Analysis: Theme co-occurrence

Which themes tend to appear together in the same answers?

In [ ]:
# Count how often each pair of themes co-occurs in the same answer
cooccurrence = pd.DataFrame(0, index=THEME_CATEGORIES, columns=THEME_CATEGORIES)

for _, row in coded_df.iterrows():
    active_themes = [t for t in THEME_CATEGORIES if row[f"theme_{t}"] == 1]
    for t1, t2 in combinations(active_themes, 2):
        cooccurrence.loc[t1, t2] += 1
        cooccurrence.loc[t2, t1] += 1

# Filter to themes that actually appear
active = [t for t in THEME_CATEGORIES if cooccurrence.loc[t].sum() > 0]
cooccurrence_filtered = cooccurrence.loc[active, active]
cooccurrence_filtered.index = [THEME_LABELS[t] for t in active]
cooccurrence_filtered.columns = [THEME_LABELS[t] for t in active]

print("Theme co-occurrence matrix (# of answers where both themes appear):\n")
display(cooccurrence_filtered.style.background_gradient(
    cmap="Blues", axis=None
).format("{:.0f}"))

# Top co-occurring pairs
print("\nTop co-occurring theme pairs:\n")
pairs = []
for t1, t2 in combinations(THEME_CATEGORIES, 2):
    count = cooccurrence.loc[t1, t2]
    if count > 0:
        pairs.append((THEME_LABELS[t1], THEME_LABELS[t2], count))

pairs.sort(key=lambda x: x[2], reverse=True)
for t1, t2, count in pairs[:10]:
    print(f"  {t1} + {t2}: {int(count)} co-occurrences")

## 9. Analysis: Participant typologies

Profiles each participant based on their dominant themes and overall sentiment.

In [ ]:
# Build a profile for each participant
profiles = []

for participant in df["_source_file"]:
    p_data = coded_df[coded_df["participant"] == participant]

    # Top themes for this participant
    theme_counts = p_data[theme_binary_cols].sum().sort_values(ascending=False)
    top_themes = [
        THEME_LABELS[t.replace("theme_", "")]
        for t in theme_counts.index
        if theme_counts[t] > 0
    ][:5]

    # Average sentiment
    avg_sent = p_data["sentiment"].dropna().mean()

    # Most common summary tags
    tags = [t for t in p_data["summary_tag"].dropna().tolist() if t]

    # Gym confidence (from the confidence question)
    confidence_row = p_data[p_data["question"].str.contains("confident", case=False)]
    confidence_sentiment = confidence_row["sentiment"].values[0] if len(confidence_row) > 0 else None

    # Barrier question
    barrier_row = p_data[p_data["question"].str.contains("barrier", case=False)]
    barrier_tag = barrier_row["summary_tag"].values[0] if len(barrier_row) > 0 else None

    profiles.append({
        "participant": participant,
        "avg_sentiment": round(avg_sent, 2) if not np.isnan(avg_sent) else None,
        "top_themes": ", ".join(top_themes),
        "primary_barrier": barrier_tag,
        "gym_confidence": confidence_sentiment,
        "num_themes_mentioned": len(top_themes),
        "sample_tags": "; ".join(tags[:5]),
    })

profiles_df = pd.DataFrame(profiles)

print("Participant profiles:\n")
for _, p in profiles_df.iterrows():
    sent_label = "positive" if p["avg_sentiment"] and p["avg_sentiment"] > 0.3 else "negative" if p["avg_sentiment"] and p["avg_sentiment"] < -0.3 else "mixed"
    print(f"  {p['participant']}")
    print(f"    Overall tone: {sent_label} ({p['avg_sentiment']:+.1f})")
    print(f"    Top themes: {p['top_themes']}")
    print(f"    Primary barrier: {p['primary_barrier']}")
    print(f"    Gym confidence: {p['gym_confidence']}")
    print()

## 10. Generate manual review spreadsheet

Exports everything to an Excel file with:
- **Sheet 1: Coded Answers** — every answer with auto-coded themes, sentiment, and a column for your manual review notes
- **Sheet 2: Theme Prevalence** — summary counts
- **Sheet 3: Sentiment Heatmap** — the pivot table
- **Sheet 4: Co-occurrence** — the matrix
- **Sheet 5: Participant Profiles** — typology summaries

In [ ]:
os.makedirs("output", exist_ok=True)
xlsx_path = "output/thematic_coding_review.xlsx"

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:

    # Sheet 1: Full coded answers with review columns
    review_df = coded_df[["participant", "question", "answer", "themes", "sentiment", "summary_tag"]].copy()
    review_df["reviewer_themes"] = ""       # Manual override column
    review_df["reviewer_sentiment"] = ""    # Manual override column
    review_df["reviewer_notes"] = ""        # Free-text notes
    review_df.to_excel(writer, index=False, sheet_name="Coded Answers")

    # Auto-fit column widths
    ws = writer.sheets["Coded Answers"]
    ws.column_dimensions["A"].width = 25
    ws.column_dimensions["B"].width = 50
    ws.column_dimensions["C"].width = 60
    ws.column_dimensions["D"].width = 35
    ws.column_dimensions["E"].width = 12
    ws.column_dimensions["F"].width = 25
    ws.column_dimensions["G"].width = 30
    ws.column_dimensions["H"].width = 15
    ws.column_dimensions["I"].width = 30

    # Sheet 2: Theme prevalence
    prevalence_data = pd.DataFrame({
        "theme": theme_prevalence.index,
        "participants_mentioning": theme_prevalence.values.astype(int),
        "total_mentions": [int(total_mentions.get(t, 0)) for t in theme_prevalence.index],
    })
    prevalence_data.to_excel(writer, index=False, sheet_name="Theme Prevalence")

    # Sheet 3: Sentiment heatmap
    sentiment_pivot.to_excel(writer, sheet_name="Sentiment Heatmap")

    # Sheet 4: Co-occurrence
    cooccurrence_filtered.to_excel(writer, sheet_name="Co-occurrence")

    # Sheet 5: Participant profiles
    profiles_df.to_excel(writer, index=False, sheet_name="Participant Profiles")

print(f"Manual review spreadsheet saved to: {xlsx_path}")
print("\nSheets:")
print("  1. Coded Answers — review and adjust themes/sentiment in the 'reviewer_*' columns")
print("  2. Theme Prevalence — which themes came up most")
print("  3. Sentiment Heatmap — sentiment scores per question per participant")
print("  4. Co-occurrence — which themes appear together")
print("  5. Participant Profiles — typology summaries")

## 11. Export clean data for visualization

Saves CSV files ready for Altair, Tableau, or any visualization tool.

In [ ]:
# Long-format coded data (best for Altair/Vega-Lite)
coded_df.to_csv("output/coded_answers_long.csv", index=False)
print("Saved: output/coded_answers_long.csv")

# Theme prevalence
prevalence_data.to_csv("output/theme_prevalence.csv", index=False)
print("Saved: output/theme_prevalence.csv")

# Sentiment pivot (wide format, good for heatmaps)
sentiment_pivot.to_csv("output/sentiment_heatmap.csv")
print("Saved: output/sentiment_heatmap.csv")

# Co-occurrence matrix
cooccurrence_filtered.to_csv("output/theme_cooccurrence.csv")
print("Saved: output/theme_cooccurrence.csv")

# Participant profiles
profiles_df.to_csv("output/participant_profiles.csv", index=False)
print("Saved: output/participant_profiles.csv")

# Theme per participant (binary matrix, good for radar charts)
participant_themes.to_csv("output/participant_theme_matrix.csv")
print("Saved: output/participant_theme_matrix.csv")

print(f"\nAll outputs saved to ./output/")

In [ ]:
# Re-export with tab separator (more robust for Tableau)
coded_df.to_csv("output/coded_answers_long.tsv", index=False, sep="\t")
print("Saved: output/coded_answers_long.tsv")

# Also export as Excel for direct Tableau connection
coded_df.to_excel("output/coded_answers_long.xlsx", index=False, engine="openpyxl")
print("Saved: output/coded_answers_long.xlsx")

In [ ]:
# Re-export all outputs as Excel for Tableau compatibility
prevalence_data.to_excel("output/theme_prevalence.xlsx", index=False, engine="openpyxl")
print("Saved: output/theme_prevalence.xlsx")

sentiment_pivot.to_excel("output/sentiment_heatmap.xlsx", engine="openpyxl")
print("Saved: output/sentiment_heatmap.xlsx")

cooccurrence_filtered.to_excel("output/theme_cooccurrence.xlsx", engine="openpyxl")
print("Saved: output/theme_cooccurrence.xlsx")

profiles_df.to_excel("output/participant_profiles.xlsx", index=False, engine="openpyxl")
print("Saved: output/participant_profiles.xlsx")

participant_themes.to_excel("output/participant_theme_matrix.xlsx", engine="openpyxl")
print("Saved: output/participant_theme_matrix.xlsx")

print("\nAll Excel files saved to ./output/")

In [ ]:
# Reshape co-occurrence matrix into long format for Tableau
cooccurrence_long = []
for t1 in cooccurrence_filtered.index:
    for t2 in cooccurrence_filtered.columns:
        val = cooccurrence_filtered.loc[t1, t2]
        if val > 0 and t1 != t2:
            cooccurrence_long.append({
                "theme_1": t1,
                "theme_2": t2,
                "count": int(val)
            })

cooccurrence_long_df = pd.DataFrame(cooccurrence_long)
cooccurrence_long_df.to_excel("output/theme_cooccurrence_long.xlsx", index=False, engine="openpyxl")
print(f"Saved: output/theme_cooccurrence_long.xlsx ({len(cooccurrence_long_df)} rows)")

In [ ]:
# Reshape participant theme matrix into long format for Tableau
participant_themes_reset = participant_themes.reset_index()
participant_long = []
for _, row in participant_themes_reset.iterrows():
    participant = row["participant"]
    for col in participant_themes.columns:
        theme_name = THEME_LABELS[col.replace("theme_", "")]
        participant_long.append({
            "participant": participant,
            "theme": theme_name,
            "present": int(row[col])
        })

participant_long_df = pd.DataFrame(participant_long)
participant_long_df.to_excel("output/participant_theme_matrix_long.xlsx", index=False, engine="openpyxl")
print(f"Saved: output/participant_theme_matrix_long.xlsx ({len(participant_long_df)} rows)")